# Notebook 03: Tabular Baseline Models

Train and evaluate tabular ML models on patient metadata (age, gender) to predict anemia severity.

## 1. Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_class_weight

sns.set_style("whitegrid")
BASE_DIR = Path("..")
CLASSES = ["Normal", "Mild", "Moderate", "Severe"]
RANDOM_STATE = 42


## 2. Load and Prepare Tabular Features

In [ ]:
df = pd.read_csv(BASE_DIR / "data" / "metadata.csv")

# Encode gender
le_gender = LabelEncoder()
df["gender_enc"] = le_gender.fit_transform(df["gender"])

# Age in years (for interpretability)
df["age_years"] = df["age"] / 12.0

# Encode target
le_label = LabelEncoder()
le_label.fit(CLASSES)
df["label"] = le_label.transform(df["diagnosis"])

# Feature matrix and target
X = df[["age_years", "gender_enc"]].values
y = df["label"].values

print(f"Features shape: {X.shape}")
print(f"Class mapping: {dict(zip(le_label.classes_, le_label.transform(le_label.classes_)))}")
print("Class distribution:")
print(pd.Series(y).value_counts().sort_index())


## 3. Stratified Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")


## 4. Class Weights

In [ ]:
classes_arr = np.unique(y_train)
cw = compute_class_weight("balanced", classes=classes_arr, y=y_train)
class_weight_dict = dict(zip(classes_arr.tolist(), cw.tolist()))
print("Class weights:", class_weight_dict)


## 5. Train Baseline Models

In [ ]:
# --- Logistic Regression ---
lr_model = LogisticRegression(
    max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE, multi_class="auto"
)
lr_model.fit(X_train_sc, y_train)

# --- Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=10, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
)
rf_model.fit(X_train_sc, y_train)

# --- XGBoost ---
xgb_model = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    use_label_encoder=False, eval_metric="mlogloss",
    random_state=RANDOM_STATE, n_jobs=-1
)
xgb_model.fit(X_train_sc, y_train)

print("Models trained successfully.")


## 6. Evaluation Helper

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te, classes_names):
    """Evaluate a sklearn model and print accuracy, macro F1, and confusion matrix."""
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    f1  = f1_score(y_te, y_pred, average="macro", zero_division=0)
    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_te, y_pred, target_names=classes_names, zero_division=0))

    cm = confusion_matrix(y_te, y_pred, labels=list(range(len(classes_names))))
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix — {name}", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"../models/saved_models/cm_tabular_{name.replace(' ', '_')}.png", bbox_inches="tight")
    plt.show()
    return acc, f1, y_pred


## 7. Compare Models

In [ ]:
results = {}
for name, model in [
    ("Logistic Regression", lr_model),
    ("Random Forest", rf_model),
    ("XGBoost", xgb_model),
]:
    acc, f1, _ = evaluate_model(name, model, X_train_sc, X_test_sc, y_train, y_test, CLASSES)
    results[name] = {"accuracy": acc, "macro_f1": f1}

results_df = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
print("\n=== Model Comparison ===")
print(results_df.round(4))


## 8. 5-Fold Cross-Validation on Best Model

In [ ]:
best_model_name = results_df.index[0]
best_model = {"Logistic Regression": lr_model, "Random Forest": rf_model, "XGBoost": xgb_model}[best_model_name]
print(f"Best model: {best_model_name}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    best_model, X, y, cv=skf,
    scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"},
    return_train_score=False, n_jobs=-1
)
print(f"CV Accuracy : {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")
print(f"CV Macro F1 : {cv_results['test_macro_f1'].mean():.4f} ± {cv_results['test_macro_f1'].std():.4f}")


## 9. Save Best Tabular Model

In [ ]:
os.makedirs("../models/saved_models", exist_ok=True)

joblib.dump(lr_model,  "../models/saved_models/lr_model.pkl")
joblib.dump(rf_model,  "../models/saved_models/rf_model.pkl")
joblib.dump(xgb_model, "../models/saved_models/xgb_model.pkl")
joblib.dump(scaler,    "../models/saved_models/scaler_tabular.pkl")
joblib.dump(le_gender, "../models/saved_models/le_gender.pkl")
joblib.dump(le_label,  "../models/saved_models/le_label.pkl")

print("All tabular models and preprocessors saved.")
